In [6]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams.update({'axes.titlesize': 'small'})

import torch

import torchvision
from torchvision.utils import make_grid

from tqdm import tqdm

import os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print(f"GPU: {torch.cuda.is_available()}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device: ", device)

GPU: True
device:  cuda


In [9]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from pathlib import Path
from tqdm import tqdm
import cv2

import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams.update({'axes.titlesize': 'small'})

import json

def hex2rgb(col_hex: str):
    col_hex = col_hex.lstrip("#")
    return tuple(int(col_hex[i:i+2], 16) for i in (0, 2, 4))

def rgb2hex(r, g, b):
    return '#{:02x}{:02x}{:02x}'.format(r, g, b)

def get_classes_from_meta(json_path):
    try:
        # Открываем и читаем JSON-файл
        with open(json_path, 'r', encoding='utf-8') as file:
            root = json.load(file)
        
        # Проверяем, есть ли ключ "classes" в "root"
        if "classes" not in root:
            raise KeyError("Ключ 'classes' отсутствует в 'root'.")
        
        # Проходим по массиву "classes"
        classes = root["classes"]
        if not isinstance(classes, list):
            raise TypeError("'classes' должен быть массивом (списком).")
        
        # Обрабатываем каждый элемент в "classes"
        class_color_map = set() # color: classs name
        
        for cls in classes:
            class_color_map.add(hex2rgb(cls['color']))
        return class_color_map
            
    except FileNotFoundError:
        print(f"Файл {json_path} не найден.")
    except json.JSONDecodeError:
        print(f"Ошибка при парсинге JSON-файла {json_path}.")
    except KeyError as e:
        print(f"Ошибка: {e}")
    except TypeError as e:
        print(f"Ошибка: {e}")
    except Exception as e:
        print(f"Произошла непредвиденная ошибка: {e}")


class Dataset(Dataset):
    def __init__(self, root_dir, path_to_meta, transform=None):
        self.root_dir = root_dir
        self.meta_classes = get_classes_from_meta(path_to_meta)
        self.transform = transform
        car_path = Path(f"{root_dir}/cars")
        mask_path = Path(f"{root_dir}/masks")

        self.car_images = []
        self.mask_images = []
        
        self.car_images.extend([p.name for p in list(car_path.glob("*.png"))])
        self.mask_images.extend([p.name for p in list(mask_path.glob("*.png"))])

        self.car_images.extend([p.name for p in list(car_path.glob("*.jpg"))])
        self.mask_images.extend([p.name for p in list(mask_path.glob("*.jpg"))])

        self.car_images.extend([p.name for p in list(car_path.glob("*.jpeg"))])
        self.mask_images.extend([p.name for p in list(mask_path.glob("*.jpeg"))])

        self.car_images = sorted(self.car_images)
        self.mask_images = sorted(self.mask_images)

    def _resize_and_pad(self, image, mask):
        h, w = image.shape[:2]
        max_size = 512
        
        # Вычисляем новые размеры с сохранением пропорций
        if h > w:
            new_h = max_size
            new_w = int(w * max_size / h)
        else:
            new_w = max_size
            new_h = int(h * max_size / w)
        
        # Изменяем размер с разными интерполяциями
        image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)
        mask = cv2.resize(mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
        
        # Создаем новые изображения с черным фоном
        padded_image = np.zeros((max_size, max_size, 3), dtype=image.dtype)
        padded_mask = np.zeros((max_size, max_size, 3), dtype=mask.dtype)
        
        # Вычисляем позиции для вставки
        top = (max_size - new_h) // 2
        left = (max_size - new_w) // 2
        
        # Вставляем изображения в центр
        padded_image[top:top+new_h, left:left+new_w] = image
        padded_mask[top:top+new_h, left:left+new_w] = mask
        
        return padded_image, padded_mask

    def split_segmentation_masks(self, mask, classes):
        """
        Разделяет трёхканальные изображения (маски сегментации) на бинарные маски по классам.
    
        :param batch: NumPy массив формы (3, H, W), где
                      H - высота изображения,
                      W - ширина изображения,
                      3 - три канала (RGB).
        :return: NumPy массив формы (N, C, H, W), где C - количество уникальных классов (цветов),
                 а каждый канал содержит бинарную маску для соответствующего класса.
        """
        # Определяем уникальные цвета во всём батче
    
        # Создаем пустой массив для бинарных масок
        H, W, _  = mask.shape
        num_classes = len(classes)
        binary_masks = np.zeros((num_classes, H, W), dtype=np.uint8)

        H, W, _ = mask.shape
    
        # Преобразуем classes в список для упорядочивания
        classes = list(classes)
        
        # Проходим по каждому цвету из classes
        for i, color in enumerate(classes):
            # Создаем бинарную маску для текущего цвета
            binary_masks[i] = np.all(mask == color, axis=-1).astype(np.uint8)
    
        return binary_masks    
        
    def __len__(self):
        return len(self.car_images)

    def __getitem__(self, idx):
        car_path = Path(self.root_dir, 'cars', self.car_images[idx])
        mask_path = Path(self.root_dir, 'masks', self.mask_images[idx]) 

        car_image = np.array(Image.open(car_path).convert('RGB'))
        mask_image = np.array(Image.open(mask_path).convert('RGB'))

        # Предобработка: изменение размера + паддинг
        car_image, mask_image = self._resize_and_pad(car_image, mask_image)
        
        # Применение преобразований
        if self.transform:
            augmented = self.transform(
                image=car_image.astype(np.float32)/255.0,
                mask=mask_image.astype(np.uint8)
            )
            car_image = augmented['image']
            mask_image = augmented['mask']

        else:
            car_image = car_image.astype(np.float32)/255.0
            mask_image = mask_image.astype(np.uint8)
            car_image = torch.from_numpy(car_image).permute(2, 0, 1).float()

        mask_image =  torch.from_numpy(self.split_segmentation_masks(np.array(mask_image), self.meta_classes))
        
        return car_image, mask_image


transform = A.Compose([
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    #A.Rotate(limit=15, p=0.5),
    #A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
], additional_targets={
        'image1': 'image'}, p=1)

transform_test = A.Compose([
    ToTensorV2(),
], additional_targets={
        'image1': 'image'}, p=1)

train_dataset = Dataset(root_dir='Car parts dataset/File1/', path_to_meta="Car parts dataset/meta.json", transform=transform)
train_dataloader = DataLoader(train_dataset, batch_size=2, shuffle=False, num_workers=0) # Для ускорения процесса обучения можно попробовать увеличить num_workers

test_dataset = Dataset(root_dir='Car parts dataset/File1/', path_to_meta="Car parts dataset/meta.json", transform=transform_test)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CrossEntropyLoss(nn.Module):
    """ 
    Custom implementation of CrossEntropyLoss.
    Args:
        pred: Tensor of shape (batch_size, num_classes, w, h) - predicted logits.
        target: Tensor of shape (batch_size, num_classes, w, h) - one-hot encoded ground truth.
    """
    
    def __init__(self):
        super().__init__()
        self.eps = 1e-8  # Small epsilon to avoid log(0)

    def forward(self, pred, target):
        # Apply log-softmax to predicted logits
        log_softmax = F.log_softmax(pred, dim=1)  # Logarithm of softmax probabilities

        # Compute element-wise product with target (one-hot encoded)
        loss = -target * log_softmax  # Negative log-likelihood

        # Sum over classes and take the mean over batch, width, and height
        return loss.sum(dim=1).mean()

In [13]:
# GRADED CELL: train_model
from skimage.metrics import mean_squared_error as mse

def train_model(model, train_dataloader, optimizer, criterion, scheduler, 
                num_epochs=200, checkpoints_path="./checkpoints", 
                use_grad_clip=True, save_checkpoints=True, load_last_checkpoint=True,
                save_epoch=1):
    
    # Создаем папку для чекпоинтов
    if save_checkpoints:
        PATH = checkpoints_path
        SAVE_EPOCH = save_epoch  
        os.makedirs(PATH, exist_ok = True)
        
    last_epoch = -1
    last_dict = None
    if load_last_checkpoint:
        for file_name in tqdm(list(os.listdir(PATH))):
            try:
                cur_dict = torch.load(os.path.join(PATH, file_name), map_location=torch.device('cpu'))
                if cur_dict['epoch'] > last_epoch:
                    last_epoch = cur_dict['epoch']
                    if last_dict is not None:
                        del last_dict
                        
                    last_dict = cur_dict
                else:
                    del cur_dict
            except:
                pass
            
        if last_dict is not None:
            model.load_state_dict(last_dict['model_state_dict'])
            optimizer.load_state_dict(last_dict['optimizer_state_dict'])
            scheduler.load_state_dict(last_dict['scheduler_state_dict'])
            last_epoch = last_dict['epoch']
            print(f'Загружен последний чекпоинт из эпохи {last_epoch}')
            del last_dict
    
    print("last_epoch:", last_epoch)

    # Цикл обучения
    for epoch in range(last_epoch + 1, num_epochs):
        total_loss = 0.0
        total_samples = 0 

        for inputs, targets in tqdm(train_dataloader, desc=f'Эпоха {epoch}'):
            optimizer.zero_grad()

            inputs = inputs.to(device)  
            targets = targets.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            loss.backward()
            
            if use_grad_clip:
                # Здесь нужно реализовать ограничение градиентов
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
                pass

            optimizer.step()
            
            total_loss += loss.item() * inputs.size(0)
            total_samples += inputs.size(0)

        avg_loss = total_loss / total_samples
        
        # Сохраняем чекпоинт каждые SAVE_EPOCH эпох
        if save_checkpoints and epoch % SAVE_EPOCH == 0:
            torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    }, os.path.join(PATH, f"epoch_{epoch}.pth"))
    
        # Обновляем learning rate
        scheduler.step()
        print(f'Эпоха {epoch}, Loss: {avg_loss} LR: {scheduler.get_last_lr()[0]}')

In [ ]:
from utils import get_scheduler
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="resnet34",       
    encoder_weights="imagenet",    # !!!!!!!!
    in_channels=3,                 
    classes=8,                    
    activation=None,               
    decoder_channels=(256, 128, 64, 32, 16)  
)

torch.manual_seed(11)
model = model.to(device)

criterion = CrossEntropyLoss() # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
use_grad_clip = False
scheduler = get_scheduler(optimizer)

train_model(model, train_dataloader, optimizer, criterion, scheduler, num_epochs=25, checkpoints_path='./checkpoints/baseline')

100%|██████████| 1/1 [00:00<00:00,  5.89it/s]


Загружен последний чекпоинт из эпохи 0
last_epoch: 0


Эпоха 1:   0%|          | 0/407 [00:00<?, ?it/s]/tmp/ipykernel_15952/3849774197.py:167: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  mask_image =  torch.from_numpy(self.split_segmentation_masks(np.array(mask_image), self.meta_classes))
Эпоха 1: 100%|██████████| 407/407 [01:07<00:00,  6.07it/s]


Эпоха 1, Loss: -5744.739917211509 LR: 0.0009755285028649954


Эпоха 2: 100%|██████████| 407/407 [01:07<00:00,  6.06it/s]


...

Эпоха 17: 100%|██████████| 407/407 [01:06<00:00,  6.16it/s]


Эпоха 17, Loss: -93224.54431962146 LR: 0.00010306265639001248


Эпоха 18: 100%|██████████| 407/407 [01:07<00:00,  6.02it/s]


Эпоха 18, Loss: -98731.89436808968 LR: 4.775527394874911e-05


Эпоха 19:  34%|███▍      | 138/407 [00:23<00:45,  5.93it/s]